# AI Workforce Intelligence Platform

# Feature Engineering

## Objective

The objective of this notebook is to transform raw employee attributes into meaningful business-oriented features that improve model performance and interpretability.

Feature engineering incorporates HR domain knowledge to derive indicators related to employee career progression, compensation, work patterns, engagement, and organizational behavior. These engineered features will be used in subsequent machine learning and deep learning models to predict employee attrition more accurately.

In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
df = pd.read_csv("..//data/interim//hr_interim.csv")

In [3]:
print(df.shape)

display(df.head())

(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


In [26]:
def min_max(series):
    return (
        (series - series.min()) /
        (series.max() - series.min())
    )

# Career Features

## Career Stage

Career stage groups employees according to their total professional experience. Employees at different stages of their careers often exhibit different promotion expectations, turnover behavior, and engagement levels.

In [4]:
career_bins = [0,5,10,20,50]

career_labels = [
    "Early Career",
    "Mid Career",
    "Experienced",
    "Senior"
]

df["CareerStage"] = pd.cut(
    df["TotalWorkingYears"],
    bins=career_bins,
    labels=career_labels,
    include_lowest=True
)

df["CareerStage"].value_counts()

CareerStage
Mid Career      607
Experienced     340
Early Career    316
Senior          207
Name: count, dtype: int64

### Promotion Delay

Instead of using raw years... Let's make a business feature.

In [5]:
df["PromotionDelay"] = np.where(
    df["YearsSinceLastPromotion"] >= 5,
    1,
    0
)

### Overtime Risk

In [6]:
df["OvertimeRisk"] = (
    df["OverTime"] == "Yes"
).astype(int)

### Frequent Traveler

In [7]:
df["FrequentTraveler"] = (
    df["BusinessTravel"] == "Travel_Frequently"
).astype(int)

### Long Commute

In [8]:
df["LongCommute"] = np.where(
    df["DistanceFromHome"] >= 15,
    1,
    0
)

### Experience Ratio

Rather than using raw experience...
Normalize it by age.

In [9]:
df["ExperienceRatio"] = (
    df["TotalWorkingYears"] /
    df["Age"]
).round(2)

In [39]:
df["ExperienceMaturity"] = pd.cut(
    df["ExperienceRatio"],
    bins=[0,0.20,0.40,0.60,1],
    labels=[
        "Beginner",
        "Developing",
        "Experienced",
        "Veteran"
    ],
    include_lowest=True
)

### Tenure Ratio

In [10]:
df["TenureRatio"] = (
    df["YearsAtCompany"] /
    df["TotalWorkingYears"]
)

df["TenureRatio"] = (
    df["TenureRatio"]
    .fillna(0)
    .round(2)
)

In [40]:
df["TenureCommitment"] = pd.cut(
    df["TenureRatio"],
    bins=[-0.01,0.25,0.50,0.75,1],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

## Engagement Score

The Engagement Score is a composite feature designed to estimate overall employee engagement by combining multiple workplace-related indicators.

Components:
- Job Involvement
- Job Satisfaction
- Environment Satisfaction
- Relationship Satisfaction

Higher scores indicate greater engagement.

In [27]:
df["EngagementScore"] = (
    min_max(df["JobInvolvement"]) +
    min_max(df["JobSatisfaction"]) +
    min_max(df["EnvironmentSatisfaction"]) +
    min_max(df["RelationshipSatisfaction"])
)

In [31]:
df["EngagementLevel"] = pd.cut(
    df["EngagementScore"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

## Wellbeing Score

The Wellbeing Score measures an employee's perceived work quality by combining work-life balance and environmental satisfaction.

In [28]:
df["WellbeingScore"] = (
    min_max(df["WorkLifeBalance"]) +
    min_max(df["EnvironmentSatisfaction"])
)

In [32]:
df["WellbeingLevel"] = pd.cut(
    df["WellbeingScore"],
    bins=[-0.01,0.5,1,1.5,2],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

### Career Growth Score

Employees usually feel positive about career growth when they:

- Have been promoted recently
- Have higher job levels
- Receive salary hikes

Why 1 -?

Because:

Recent promotion = good
Long wait = bad

So we invert the normalized value.

In [33]:
career_growth = (
    min_max(df["JobLevel"]) +
    min_max(df["PercentSalaryHike"]) +
    (1 - min_max(df["YearsSinceLastPromotion"]))
)

df["CareerGrowthScore"] = (
    career_growth / 3
).round(3)

In [34]:
df["CareerGrowthLevel"] = pd.cut(
    df["CareerGrowthScore"],
    bins=[-0.01,0.25,0.50,0.75,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Excellent"
    ]
)

### Loyalty Score

Employees who have spent most of their career at one company are generally more loyal. Adding +1 avoids division by zero for employees with no prior work experience.

In [14]:
df["LoyaltyScore"] = (
    df["YearsAtCompany"] /
    (df["TotalWorkingYears"] + 1)
).round(2)

In [37]:
df["LoyaltyLevel"] = pd.cut(
    df["LoyaltyScore"],
    bins=[-0.01,0.25,0.50,0.75,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

### Manager Stability

Employees with longer relationships with the same manager may have different retention patterns than those who change managers frequently.

In [15]:
df["ManagerStability"] = (
    df["YearsWithCurrManager"] /
    (df["YearsAtCompany"] + 1)
).round(2)

In [38]:
df["ManagerStabilityLevel"] = pd.cut(
    df["ManagerStability"],
    bins=[-0.01,0.25,0.50,0.75,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

### Income Band

Rather than using raw salary, create business-friendly categories.

In [16]:
df["IncomeBand"] = pd.qcut(
    df["MonthlyIncome"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

### Salary Percentile

In [17]:
df["SalaryPercentile"] = (
    df["MonthlyIncome"]
    .rank(pct=True)
    .round(3)
)

### Income Per Job Level

Instead of looking only at salary...
Normalize salary by seniority.

In [18]:
df["IncomePerJobLevel"] = (
    df["MonthlyIncome"] /
    df["JobLevel"]
).round(2)

### Young Professional

In [19]:
df["YoungProfessional"] = (
    df["Age"] <= 30
).astype(int)

### Senior Employee

In [20]:
df["SeniorEmployee"] = (
    df["Age"] >= 45
).astype(int)

### Commute Band

In [21]:
df["CommuteBand"] = pd.cut(
    df["DistanceFromHome"],
    bins=[0,5,15,30],
    labels=[
        "Near",
        "Moderate",
        "Far"
    ],
    include_lowest=True
)

### Experience Band

In [22]:
df["ExperienceBand"] = pd.cut(
    df["TotalWorkingYears"],
    bins=[0,5,10,20,40],
    labels=[
        "Junior",
        "Intermediate",
        "Senior",
        "Expert"
    ],
    include_lowest=True
)

### Attrition Risk Score (Business Heuristic)

This is not the model prediction. It's a simple rule-based score that captures common risk factors and can later be compared with the machine learning model.

Higher scores indicate employees who may warrant additional attention.

In [35]:
risk = (
    min_max(df["OvertimeRisk"]) +
    min_max(df["PromotionDelay"]) +
    min_max(df["LongCommute"]) +
    (1 - min_max(df["WorkLifeBalance"])) +
    (1 - min_max(df["JobSatisfaction"]))
)

df["AttritionRiskScore"] = (
    risk / 5
).round(3)

In [36]:
df["AttritionRiskLevel"] = pd.cut(
    df["AttritionRiskScore"],
    bins=[-0.01,0.25,0.50,0.75,1],
    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk",
        "Critical Risk"
    ]
)

In [24]:
engineered_features = [
    "CareerStage",
    "PromotionDelay",
    "OvertimeRisk",
    "FrequentTraveler",
    "LongCommute",
    "ExperienceRatio",
    "TenureRatio",
    "EngagementScore",
    "WellbeingScore",
    "CareerGrowthScore",
    "LoyaltyScore",
    "ManagerStability",
    "IncomeBand",
    "SalaryPercentile",
    "IncomePerJobLevel",
    "YoungProfessional",
    "SeniorEmployee",
    "CommuteBand",
    "ExperienceBand",
    "AttritionRiskScore"
]

display(df[engineered_features].head())

,CareerStage,PromotionDelay,OvertimeRisk,FrequentTraveler,LongCommute,ExperienceRatio,TenureRatio,EngagementScore,WellbeingScore,CareerGrowthScore,LoyaltyScore,ManagerStability,IncomeBand,SalaryPercentile,IncomePerJobLevel,YoungProfessional,SeniorEmployee,CommuteBand,ExperienceBand,AttritionRiskScore
0,Mid Career,0,1,0,0,0.20,0.75,10,3,13,0.67,0.71,High,0.62,2996.50,0,0,Near,Intermediate,6
1,Mid Career,0,0,1,0,0.20,1.00,11,6,24,0.91,0.64,High,0.52,2565.00,0,1,Moderate,Intermediate,3
2,Mid Career,0,1,0,0,0.19,0.00,11,7,16,0.00,0.00,Low,0.05,2090.00,0,0,Near,Intermediate,5
3,Mid Career,0,1,1,0,0.24,1.00,13,7,9,0.89,0.00,Low,0.25,2909.00,0,0,Near,Intermediate,5
4,Mid Career,0,0,0,0,0.22,0.33,10,4,11,0.29,0.67,Medium,0.32,3468.00,1,0,Near,Intermediate,3


In [41]:
engineered_columns = [
    "EngagementScore", "EngagementLevel",
    "WellbeingScore", "WellbeingLevel",
    "CareerGrowthScore", "CareerGrowthLevel",
    "AttritionRiskScore", "AttritionRiskLevel",
    "LoyaltyScore", "LoyaltyLevel",
    "ManagerStability", "ManagerStabilityLevel",
    "ExperienceRatio", "ExperienceMaturity",
    "TenureRatio", "TenureCommitment"
]

display(df[engineered_columns].head())

,EngagementScore,EngagementLevel,WellbeingScore,WellbeingLevel,CareerGrowthScore,CareerGrowthLevel,AttritionRiskScore,AttritionRiskLevel,LoyaltyScore,LoyaltyLevel,ManagerStability,ManagerStabilityLevel,ExperienceRatio,ExperienceMaturity,TenureRatio,TenureCommitment
0,2.00,Medium,0.33,Low,0.42,Medium,0.40,Moderate Risk,0.67,High,0.71,High,0.20,Beginner,0.75,High
1,2.33,High,1.33,High,0.68,High,0.20,Low Risk,0.91,Very High,0.64,High,0.20,Beginner,1.00,Very High
2,2.33,High,1.67,Very High,0.43,Medium,0.33,Moderate Risk,0.00,Low,0.00,Low,0.19,Beginner,0.00,Low
3,3.00,High,1.67,Very High,0.27,Medium,0.33,Moderate Risk,0.89,Very High,0.00,Low,0.24,Developing,1.00,Very High
4,2.00,Medium,0.67,Medium,0.31,Medium,0.20,Low Risk,0.29,Medium,0.67,High,0.22,Developing,0.33,Moderate


In [43]:
from pathlib import Path

processed_file = Path("../data/processed/hr_feature_engineered.csv")

df.to_csv(
    processed_file,
    index=False
)

print(f"Feature-engineered dataset saved to: {processed_file}")

Feature-engineered dataset saved to: ..\data\processed\hr_feature_engineered.csv
